In [1]:
pip install torch transformers

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [3]:
model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((2048,), eps=1e-06)
    (ro

In [15]:
def generate_response(question, instruction, max_new_tokens=1024):
    messages = [
        {
            "role": "system",
            "content": instruction
        },
        {
            "role": "user",
            "content": question
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

In [16]:
question = "Why is the sky blue?"

ans_concise = generate_response(
        question,
        "Answer concisely in 1–2 sentences."
    )

ans_detailed = generate_response(
        question,
        "Give a detailed and thorough explanation."
    )

ans_noinstruction = generate_response(
        question,
        ""
    )

In [17]:
print(ans_concise)
print("Lenght:", len(ans_concise), "Words:", len(ans_concise.strip().split()))
print("=====================================")
print(ans_detailed)
print("Lenght:", len(ans_detailed), "Words:", len(ans_detailed.strip().split()))
print("=====================================")
print(ans_noinstruction)
print("Lenght:", len(ans_noinstruction), "Words:", len(ans_noinstruction.strip().split()))

The sky appears blue because molecules of gas in Earth's atmosphere scatter sunlight more efficiently for blue light, which then travels back to our eyes.
Lenght: 154 Words: 25
The sky appears blue due to a phenomenon known as Rayleigh scattering, which is named after Lord Rayleigh who first explained this effect in 1871. This process occurs when sunlight enters the Earth's atmosphere and collides with molecules of air and tiny particles like dust and water droplets.

Here’s a detailed explanation of why the sky is blue:

### 1. Composition of Sunlight:
Sunlight is actually a mixture of all colors of light, each with a different wavelength. When sunlight enters the Earth's atmosphere, it encounters gas molecules, primarily nitrogen (N₂) and oxygen (O₂), and smaller amounts of other gases and particles.

### 2. Rayleigh Scattering:
Rayleigh scattering is a type of scattering that occurs when light interacts with particles much smaller than the wavelength of the light. In the case of the

In [21]:
import json

with open("prompts.json", "r") as f:
    prompts = json.load(f)

In [22]:
results = []

for item in prompts:
    question = item["question"]

    concise = generate_response(
        question,
        "Answer concisely in 1–2 sentences.",
    )

    concise2 = generate_response(
        question,
        "Answer concisely in a few sentences.",
    )

    detailed = generate_response(
        question,
        "Give a detailed and thorough explanation.",
    )

    results.append({
        "id": item["id"],
        "question": question,
        "concise": concise,
        "concise2": concise2,
        "detailed": detailed,
        "concise_words": len(concise.strip().split()),
        "concise2_words": len(concise2.strip().split()),
        "detailed_words": len(detailed.strip().split())
    })

In [23]:
import pandas as pd

df = pd.DataFrame(results)

df[[
    "id",
    "concise_words",
    "concise2_words",
    "detailed_words"
]]

,id,concise_words,concise2_words,detailed_words
0,1,25,26,535
1,2,31,33,508
2,3,28,47,487
3,4,25,27,378
4,5,30,46,657
5,6,32,32,673
6,7,28,26,604
7,8,30,30,476
8,9,26,43,575
9,10,36,36,527


In [24]:
print("Average concise:",
      df["concise_words"].mean())

print("Average concise:",
      df["concise2_words"].mean())

print("Average detailed:",
      df["detailed_words"].mean())

Average concise: 28.1
Average concise: 36.166666666666664
Average detailed: 537.6333333333333


In [25]:
print(
    "Average ratio1:",
    df["detailed_words"].mean() /
    df["concise_words"].mean()
)
print(
    "Average ratio2:",
    df["detailed_words"].mean() /
    df["concise2_words"].mean()
)

Average ratio1: 19.13285883748517
Average ratio2: 14.865437788018435


In [26]:
df.to_csv("baseline_responses.csv", index=False)